# MVP — Pipeline de Engenharia de Dados na Nuvem (Databricks Lakehouse)
## Disciplina: Sistemas de Apoio à Decisão (PSP8) — Engenharia de Produção / UnB
**Aluno:** Rodrigo Fregonasse  
**Professor:** Dr. André Luiz Marques Serrano  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist (Kaggle)

---
### Arquitetura Implementada: Data Lakehouse (Medalhão)
Este notebook implementa a arquitetura de dados completa na plataforma de nuvem **Databricks**:
1. **Criação do Banco de Dados Analítico:**  no catálogo/metastore do Databricks.
2. **Camada Bronze (Data Lake Bruto):** Ingestão dos CSVs no *Databricks File System* () e persistência das tabelas brutas Delta.
3. **Camada Silver (Conformed / Cleaned):** Limpeza, tipagem estrita de datas, remoção de nulos críticos e deduplicação de avaliações via PySpark.
4. **Camada Gold (Data Warehouse Dimensional):** Modelagem em **Esquema Estrela (*Star Schema*)** com Tabela Fato e 5 Dimensões Conformes, calculando métricas de Lead Time, desvios de prazo e satisfação.
5. **Persistência em Nuvem:** Tabelas gerenciadas (*Managed Tables*) em formato **Delta Lake** com transações ACID.

In [ ]:
# 1. Instalação de Dependências no Driver do Cluster Databricks
%pip install kagglehub pandas numpy matplotlib seaborn --quiet

### 2. Criação do Banco de Dados Analítico no Databricks
Criamos formalmente o schema/database  no catálogo da nuvem para armazenar todas as tabelas do projeto.

In [ ]:
import os
import shutil
import kagglehub

# 1. Download direto via kagglehub no nó local
raw_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print(f"Dados baixados pelo kagglehub em: {raw_path}")

# 2. Copiar para o Databricks /Workspace (permitido pelo Databricks Serverless / Unity Catalog)
target_dir = "/Workspace/Shared/olist_data"
try:
    os.makedirs(target_dir, exist_ok=True)
except Exception:
    user_email = spark.sql("SELECT current_user()").collect()[0][0]
    target_dir = f"/Workspace/Users/{user_email}/olist_data"
    os.makedirs(target_dir, exist_ok=True)

for fname in os.listdir(raw_path):
    if fname.endswith(".csv"):
        shutil.copy(os.path.join(raw_path, fname), os.path.join(target_dir, fname))

print(f"Arquivos CSV disponibilizados no Workspace Databricks em: {target_dir}")

### 3. Ingestão e Persistência no Data Lake (DBFS / Camada Bronze)
Baixamos os microdados oficiais da Olist e salvamos no armazenamento persistente do Databricks ().

In [ ]:
import os
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Localizar o diretório de dados no /Workspace
target_dir = "/Workspace/Shared/olist_data"
if not os.path.exists(target_dir):
    user_email = spark.sql("SELECT current_user()").collect()[0][0]
    target_dir = f"/Workspace/Users/{user_email}/olist_data"

base_path = f"{target_dir}/"

try:
    # Leitura nativa PySpark a partir do /Workspace
    df_orders = spark.read.csv(base_path + "olist_orders_dataset.csv", header=True, inferSchema=True)
    df_items = spark.read.csv(base_path + "olist_order_items_dataset.csv", header=True, inferSchema=True)
    df_customers = spark.read.csv(base_path + "olist_customers_dataset.csv", header=True, inferSchema=True)
    df_sellers = spark.read.csv(base_path + "olist_sellers_dataset.csv", header=True, inferSchema=True)
    df_products = spark.read.csv(base_path + "olist_products_dataset.csv", header=True, inferSchema=True)
    df_reviews = spark.read.csv(base_path + "olist_order_reviews_dataset.csv", header=True, inferSchema=True)
    df_trans = spark.read.csv(base_path + "product_category_name_translation.csv", header=True, inferSchema=True)
except Exception as e:
    print(f"Leitura nativa ajustada para leitura resiliente via Pandas: {e}")
    import pandas as pd
    def read_csv_spark(filename):
        fpath = os.path.join(base_path, filename)
        if not os.path.exists(fpath):
            fpath = os.path.join(raw_path, filename)
        pdf = pd.read_csv(fpath)
        pdf = pdf.where(pd.notnull(pdf), None)
        return spark.createDataFrame(pdf)
    
    df_orders = read_csv_spark("olist_orders_dataset.csv")
    df_items = read_csv_spark("olist_order_items_dataset.csv")
    df_customers = read_csv_spark("olist_customers_dataset.csv")
    df_sellers = read_csv_spark("olist_sellers_dataset.csv")
    df_products = read_csv_spark("olist_products_dataset.csv")
    df_reviews = read_csv_spark("olist_order_reviews_dataset.csv")
    df_trans = read_csv_spark("product_category_name_translation.csv")

# Persistir tabelas na Camada Bronze (Delta Lake)
df_orders.write.format("delta").mode("overwrite").saveAsTable("psp8_olist_dw.bronze_orders")
df_items.write.format("delta").mode("overwrite").saveAsTable("psp8_olist_dw.bronze_order_items")
df_customers.write.format("delta").mode("overwrite").saveAsTable("psp8_olist_dw.bronze_customers")
df_sellers.write.format("delta").mode("overwrite").saveAsTable("psp8_olist_dw.bronze_sellers")
df_products.write.format("delta").mode("overwrite").saveAsTable("psp8_olist_dw.bronze_products")
df_reviews.write.format("delta").mode("overwrite").saveAsTable("psp8_olist_dw.bronze_reviews")

print(f"Tabelas Bronze persistidas no catálogo psp8_olist_dw: {df_orders.count():,} pedidos brutos.")

### 4. Carga das Tabelas Bronze via PySpark e Persistência Delta
Lemos os dados do DBFS com inferência de schema e salvamos como tabelas Delta da Camada Bronze.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Leitura direta dos arquivos já baixados no cluster
df_orders = spark.read.csv(f"file:{raw_path}/olist_orders_dataset.csv", header=True, inferSchema=True)
df_items = spark.read.csv(f"file:{raw_path}/olist_order_items_dataset.csv", header=True, inferSchema=True)
df_customers = spark.read.csv(f"file:{raw_path}/olist_customers_dataset.csv", header=True, inferSchema=True)
df_sellers = spark.read.csv(f"file:{raw_path}/olist_sellers_dataset.csv", header=True, inferSchema=True)
df_products = spark.read.csv(f"file:{raw_path}/olist_products_dataset.csv", header=True, inferSchema=True)
df_reviews = spark.read.csv(f"file:{raw_path}/olist_order_reviews_dataset.csv", header=True, inferSchema=True)
df_trans = spark.read.csv(f"file:{raw_path}/product_category_name_translation.csv", header=True, inferSchema=True)

# Persistir tabelas na Camada Bronze (Delta Lake)
df_orders.write.format('delta').mode('overwrite').saveAsTable('psp8_olist_dw.bronze_orders')
df_items.write.format('delta').mode('overwrite').saveAsTable('psp8_olist_dw.bronze_order_items')
df_customers.write.format('delta').mode('overwrite').saveAsTable('psp8_olist_dw.bronze_customers')
df_sellers.write.format('delta').mode('overwrite').saveAsTable('psp8_olist_dw.bronze_sellers')
df_products.write.format('delta').mode('overwrite').saveAsTable('psp8_olist_dw.bronze_products')
df_reviews.write.format('delta').mode('overwrite').saveAsTable('psp8_olist_dw.bronze_reviews')

print(f'Tabelas Bronze persistidas no catálogo psp8_olist_dw: {df_orders.count():,} pedidos brutos.')

### 5. Camada Silver: Limpeza, Tipagem e Deduplicação
Tratamos timestamps, removemos inconsistências físicas e preparamos os atributos essenciais.

In [ ]:
# Filtrar pedidos entregues e converter timestamps
df_orders_delivered = df_orders.filter(F.col('order_status') == 'delivered')     .filter(F.col('order_delivered_customer_date').isNotNull())     .withColumn('ts_compra', F.to_timestamp('order_purchase_timestamp'))     .withColumn('ts_aprovacao', F.to_timestamp('order_approved_at'))     .withColumn('ts_despacho', F.to_timestamp('order_delivered_carrier_date'))     .withColumn('ts_entrega_real', F.to_timestamp('order_delivered_customer_date'))     .withColumn('ts_prazo_estimado', F.to_timestamp('order_estimated_delivery_date'))

# Deduplicar avaliações (manter a mais recente por pedido)
from pyspark.sql.window import Window
w_rev = Window.partitionBy('order_id').orderBy(F.col('review_answer_timestamp').desc())
df_reviews_dedup = df_reviews.withColumn('rn', F.row_number().over(w_rev))     .filter(F.col('rn') == 1)     .select('order_id', 'review_score')

print(f'Pedidos entregues válidos na Silver: {df_orders_delivered.count():,}')

### 6. Camada Gold: Construção das Dimensões Conformes
Construímos as Dimensões Clientes, Vendedores, Produtos, Rotas Logísticas e Tempo.

In [ ]:
# Dicionário de UFs para Regiões IBGE
regioes_map = F.create_map([F.lit(x) for x in [
    'SP', 'Sudeste', 'RJ', 'Sudeste', 'MG', 'Sudeste', 'ES', 'Sudeste',
    'PR', 'Sul', 'SC', 'Sul', 'RS', 'Sul',
    'DF', 'Centro-Oeste', 'GO', 'Centro-Oeste', 'MT', 'Centro-Oeste', 'MS', 'Centro-Oeste',
    'BA', 'Nordeste', 'PE', 'Nordeste', 'CE', 'Nordeste', 'MA', 'Nordeste',
    'PB', 'Nordeste', 'RN', 'Nordeste', 'AL', 'Nordeste', 'PI', 'Nordeste', 'SE', 'Nordeste',
    'AM', 'Norte', 'PA', 'Norte', 'RO', 'Norte', 'TO', 'Norte',
    'AC', 'Norte', 'AP', 'Norte', 'RR', 'Norte'
]])

# Dimensão Clientes
dim_clientes = df_customers.select(
    F.col('customer_id').alias('cliente_id'),
    F.col('customer_unique_id').alias('cliente_hash_unico'),
    F.col('customer_zip_code_prefix').alias('cep_prefixo_cliente'),
    F.col('customer_city').alias('cidade_cliente'),
    F.col('customer_state').alias('estado_cliente'),
    regioes_map[F.col('customer_state')].alias('regiao_cliente')
)

# Dimensão Vendedores
dim_vendedores = df_sellers.select(
    F.col('seller_id').alias('vendedor_id'),
    F.col('seller_zip_code_prefix').alias('cep_prefixo_vendedor'),
    F.col('seller_city').alias('cidade_vendedor'),
    F.col('seller_state').alias('estado_vendedor'),
    regioes_map[F.col('seller_state')].alias('regiao_vendedor')
)

# Dimensão Produtos
dim_produtos = df_products.join(df_trans, on='product_category_name', how='left')     .select(
        F.col('product_id').alias('produto_id'),
        F.coalesce(F.col('product_category_name'), F.lit('outros')).alias('categoria_pt'),
        F.coalesce(F.col('product_category_name_english'), F.lit('other')).alias('categoria_en'),
        F.coalesce(F.col('product_weight_g'), F.lit(1000)).alias('peso_gramas'),
        (F.coalesce(F.col('product_length_cm'), F.lit(20)) *
         F.coalesce(F.col('product_height_cm'), F.lit(15)) *
         F.coalesce(F.col('product_width_cm'), F.lit(15))).alias('volume_cm3')
    )     .withColumn('faixa_peso', F.when(F.col('peso_gramas') < 1000, '1. Leve (<1kg)')
                .when(F.col('peso_gramas') <= 5000, '2. Médio (1-5kg)')
                .when(F.col('peso_gramas') <= 15000, '3. Pesado (5-15kg)')
                .otherwise('4. Muito Pesado (>15kg)'))

# Dimensão Tempo
dim_tempo = df_orders_delivered.select(
    F.date_format('ts_compra', 'yyyyMMdd').cast('int').alias('data_id'),
    F.to_date('ts_compra').alias('data_completa'),
    F.year('ts_compra').alias('ano'),
    F.month('ts_compra').alias('mes'),
    F.dayofmonth('ts_compra').alias('dia'),
    F.date_format('ts_compra', 'EEEE').alias('dia_semana'),
    F.quarter('ts_compra').alias('trimestre')
).dropDuplicates(['data_id'])

### 7. Camada Gold: Tabela Fato e Dimensão Rotas Logísticas
Unificamos os dados e calculamos métricas de Lead Time, desvios e indicadores de atraso.

In [ ]:
fato = df_items.join(df_orders_delivered, on='order_id', how='inner')     .join(df_customers.select('customer_id', 'customer_state'), on='customer_id', how='left')     .join(df_sellers.select('seller_id', 'seller_state'), on='seller_id', how='left')     .join(df_reviews_dedup, on='order_id', how='left')

# Fato Entregas de Pedidos
fato_entregas = fato.select(
    F.col('order_id').alias('pedido_id'),
    F.col('order_item_id').alias('item_id'),
    F.col('customer_id').alias('cliente_id'),
    F.col('seller_id').alias('vendedor_id'),
    F.col('product_id').alias('produto_id'),
    F.date_format('ts_compra', 'yyyyMMdd').cast('int').alias('data_pedido_id'),
    F.concat(F.col('seller_state'), F.lit('-'), F.col('customer_state')).alias('rota_id'),
    F.col('price').alias('valor_preco'),
    F.col('freight_value').alias('valor_frete'),
    (F.col('price') + F.col('freight_value')).alias('valor_total'),
    F.round(F.col('freight_value') / F.col('price'), 4).alias('razao_frete_preco'),
    F.round((F.col('ts_entrega_real').cast('long') - F.col('ts_compra').cast('long')) / 86400.0, 2).alias('lead_time_real_dias'),
    F.round((F.col('ts_prazo_estimado').cast('long') - F.col('ts_compra').cast('long')) / 86400.0, 2).alias('prazo_prometido_dias'),
    F.round((F.col('ts_entrega_real').cast('long') - F.col('ts_prazo_estimado').cast('long')) / 86400.0, 2).alias('dias_desvio_prazo'),
    F.col('review_score')
) .withColumn('flag_atrasado', F.when(F.col('dias_desvio_prazo') > 0, 1).otherwise(0)) .withColumn('flag_atraso_grave_48h', F.when(F.col('dias_desvio_prazo') > 2.0, 1).otherwise(0)) .withColumn('flag_detrator', F.when((F.col('review_score').isNotNull()) & (F.col('review_score') <= 2), 1).otherwise(0))

# Dimensão Rotas Logísticas
dim_rotas = fato.select(
    F.concat(F.col('seller_state'), F.lit('-'), F.col('customer_state')).alias('rota_id'),
    F.col('seller_state').alias('estado_origem'),
    F.col('customer_state').alias('estado_destino'),
    regioes_map[F.col('seller_state')].alias('regiao_origem'),
    regioes_map[F.col('customer_state')].alias('regiao_destino')
).dropDuplicates(['rota_id']) .withColumn('tipo_rota', F.when(F.col('estado_origem') == F.col('estado_destino'), '1. Intraestadual')
            .when(F.col('regiao_origem') == F.col('regiao_destino'), '2. Interestadual (Mesma Região)')
            .otherwise('3. Inter-regional'))

### 8. Persistência Final no Catálogo Delta Lake (Data Warehouse / Lakehouse)
Persistimos todas as tabelas dimensionais no banco  com gerenciamento ACID completo.

In [ ]:
# Persistindo Tabelas Fato e Dimensões
fato_entregas.write.format('delta').mode('overwrite').saveAsTable('psp8_olist_dw.fato_entregas_pedidos')
dim_clientes.write.format('delta').mode('overwrite').saveAsTable('psp8_olist_dw.dim_clientes')
dim_vendedores.write.format('delta').mode('overwrite').saveAsTable('psp8_olist_dw.dim_vendedores')
dim_produtos.write.format('delta').mode('overwrite').saveAsTable('psp8_olist_dw.dim_produtos')
dim_rotas.write.format('delta').mode('overwrite').saveAsTable('psp8_olist_dw.dim_rotas_logisticas')
dim_tempo.write.format('delta').mode('overwrite').saveAsTable('psp8_olist_dw.dim_tempo')

print('Sucesso: Banco psp8_olist_dw totalmente populado e persistido em Delta Lake no Databricks!')

### 9. Validação e Consultas via Spark SQL
Demonstração das tabelas criadas no catálogo e consulta executiva.

In [ ]:
# Exibir tabelas do catálogo psp8_olist_dw
%sql
SHOW TABLES IN psp8_olist_dw;

In [ ]:
# Consulta analítica na tabela fato
%sql
SELECT 
    flag_atrasado, 
    COUNT(*) as total_itens, 
    ROUND(AVG(lead_time_real_dias), 2) as lead_time_medio,
    ROUND(AVG(review_score), 2) as review_medio,
    ROUND(AVG(flag_detrator) * 100, 2) as pct_detratores
FROM psp8_olist_dw.fato_entregas_pedidos
GROUP BY flag_atrasado;